# Notebook 3: Evaluate, Comply, Deploy

**Phase 5** | ~40 minutes

In this notebook you will:
1. Evaluate your trained YOLO26n model with standard detection metrics (mAP50)
2. Analyze *where* your model fails — object size, per-class, confidence thresholds
3. **Design a compliance algorithm** — the core challenge of this workshop
4. Benchmark inference speed — can your model run at 30+ FPS?
5. Compare approaches and reflect on the full pipeline

---

### Recap

You have completed the full distillation pipeline:
- **NB01**: Explored the dataset, ran a zero-shot baseline with YOLOe-26n, auto-labeled with a foundation model
- **NB02**: Inspected and curated labels, filtered noise, trained a YOLO26n student model

Now comes the question that matters: **is your model good enough to deploy, and can it answer the business question?**

> **AI assistance**: Use your preferred AI coding tool for guidance. See `cv_copilot_skill.md` for a full system prompt, or `cv_copilot_prompt.md` for a shorter version you can paste into ChatGPT, Claude, or any AI assistant.

---
## Section 0: The Final Challenge

Your model detects two classes: **hardhat** and **person**.

The construction site manager does not care about mAP. They care about one thing:

> **"What percentage of workers on this site are NOT wearing a hardhat?"**

Your task in this notebook: **design an algorithm** that takes your model's detections (bounding boxes for hardhats and persons) and outputs a per-image compliance assessment.

Think about it before you scroll ahead:
- You have person bounding boxes and hardhat bounding boxes.
- How do you determine which hardhats belong to which persons?
- What spatial relationship would you check?
- Where on a person's body would you expect to find a hardhat?

The evaluation sections come first (Sections 1-4) to help you understand your model's strengths and weaknesses. **Section 5 is the challenge** where you build the compliance algorithm.

Let's start by understanding how well your model actually detects things.

In [ ]:
import sys
import time
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Auto-detect device: CUDA > MPS > CPU
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
          else "cpu")
)
print(f"Device: {DEVICE}")

WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"
IMAGES_DIR = DATA_DIR / "synthetic_ppe"

# ======================================================================
# SET YOUR PATHS HERE
# ======================================================================

# Your trained model weights (from Notebook 2 training)
# Example: DATA_DIR / "ppe_results" / "my_experiment" / "weights" / "best.pt"
MODEL_PATH = DATA_DIR / "ppe_results" / "my_experiment" / "weights" / "best.pt"

# Your 2-class labeled dataset (filtered version from NB02)
EVAL_DATASET = DATA_DIR / "ppe_dataset_sam3_exp_a_filtered"

# ======================================================================
# VERIFY PATHS
# ======================================================================
for name, path in [("Model", MODEL_PATH), ("Eval dataset", EVAL_DATASET)]:
    status = "found" if path.exists() else "NOT FOUND"
    print(f"{name}: {path}  [{status}]")

if not MODEL_PATH.exists():
    print("\nModel not found. Options:")
    print("  1. Update MODEL_PATH to point to your trained weights")
    print("  2. Check data/ppe_results/ for pre-baked results")
    results_dir = DATA_DIR / "ppe_results"
    if results_dir.exists():
        for d in sorted(results_dir.iterdir()):
            best = d / "weights" / "best.pt"
            if best.exists():
                print(f"     -> {best}")


---
## Section 1: Load Model & Run Detection Evaluation

~5 minutes

Standard COCO-protocol evaluation using Ultralytics `model.val()`:
- **mAP50**: Mean Average Precision at IoU >= 0.5
- **mAP50-95**: Averaged over IoU thresholds 0.50 to 0.95
- **Per-class AP**: How well does the model detect each class?

**Expected output**: Your mAP50 should be in the **0.55 - 0.65** range if you trained on filtered SAM3 labels. If you used Qwen 3.5 labels, expect 0.50 - 0.60.

In [ ]:
from ultralytics import YOLO

# Load your trained model
model = YOLO(str(MODEL_PATH))
print(f"Model loaded: {MODEL_PATH.name}")
print(f"Classes: {model.names}")

In [ ]:
# Run COCO-style validation on your evaluation dataset
dataset_yaml = EVAL_DATASET / "dataset.yaml"
if not dataset_yaml.exists():
    dataset_yaml = EVAL_DATASET / "data.yaml"

assert dataset_yaml.exists(), f"Dataset YAML not found in {EVAL_DATASET}"

metrics = model.val(
    data=str(dataset_yaml),
    conf=0.25,
    iou=0.5,
    verbose=False,
)

# Print results
print("=" * 60)
print("  DETECTION EVALUATION RESULTS")
print("=" * 60)
print(f"  mAP50:      {metrics.box.map50:.4f}")
print(f"  mAP50-95:   {metrics.box.map:.4f}")
print()

# Per-class breakdown
class_names = model.names
print(f"  {'Class':<15} {'AP50':>8} {'Precision':>10} {'Recall':>8}")
print(f"  {'-'*15} {'-'*8} {'-'*10} {'-'*8}")
for i, name in class_names.items():
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
    print(f"  {name:<15} {ap50:>8.3f} {p:>10.3f} {r:>8.3f}")

print("=" * 60)

### Interpreting Your Results

| Metric | What it means |
|--------|---------------|
| **mAP50 > 0.60** | Good — your auto-labels and curation paid off |
| **mAP50 0.50-0.60** | Decent — room for improvement in labels or data |
| **mAP50 < 0.50** | Something went wrong — check label quality |
| **Person AP >> Hardhat AP** | Expected — persons are larger and more consistent |
| **Low recall** | Model misses objects — likely small or occluded ones |
| **Low precision** | Model hallucinates — too many false detections |

---
## Section 2: Object Size vs Detection Performance

~5 minutes

Auto-labelers and student models both struggle with very small objects.
Let's check: **are the objects your model misses systematically smaller than the ones it detects?**

This analysis loads ground truth labels and model predictions, matches them at IoU >= 0.5,
and compares the bounding box areas of true positives (detected) vs false negatives (missed).

In [ ]:
# ── Utility: Parse YOLO labels and match predictions ─────────────────────────

def parse_yolo_labels(label_path):
    """Parse a YOLO label file into list of (cls_id, cx, cy, w, h)."""
    labels = []
    if not Path(label_path).exists():
        return labels
    for line in Path(label_path).read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_id = int(parts[0])
            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            labels.append((cls_id, cx, cy, w, h))
    return labels


def compute_iou(box_a, box_b):
    """IoU between two [x1, y1, x2, y2] boxes."""
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def yolo_to_xyxy(cx, cy, w, h, img_w=1.0, img_h=1.0):
    """Convert YOLO normalized (cx, cy, w, h) to (x1, y1, x2, y2)."""
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return [x1, y1, x2, y2]

In [ ]:
# ── Size analysis: GT boxes matched vs missed ────────────────────────────────

val_images_dir = EVAL_DATASET / "images" / "val"
val_labels_dir = EVAL_DATASET / "labels" / "val"

assert val_images_dir.exists(), f"Validation images not found: {val_images_dir}"

tp_areas = []  # areas of GT boxes that were detected (TP)
fn_areas = []  # areas of GT boxes that were missed (FN)
fp_areas = []  # areas of predicted boxes with no GT match (FP)
tp_pred_areas = []  # areas of predicted boxes that matched GT (TP)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
val_images = sorted([p for p in val_images_dir.iterdir() if p.suffix.lower() in image_extensions])

print(f"Running predictions on {len(val_images)} validation images...")

for img_path in val_images:
    # Get image dimensions (once per image)
    pil_img = Image.open(img_path)
    iw, ih = pil_img.size

    # Get GT labels
    label_path = val_labels_dir / f"{img_path.stem}.txt"
    gt_labels = parse_yolo_labels(label_path)
    gt_boxes = [(cls_id, yolo_to_xyxy(cx, cy, w, h)) for cls_id, cx, cy, w, h in gt_labels]
    gt_areas_raw = [w * h for _, _, _, w, h in gt_labels]  # normalized area

    # Get predictions
    results = model.predict(str(img_path), conf=0.25, device=DEVICE, verbose=False)
    pred_boxes = []
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            xyxy = box.xyxy[0].cpu().numpy()
            # Normalize to [0,1] for area comparison
            norm_area = ((xyxy[2] - xyxy[0]) / iw) * ((xyxy[3] - xyxy[1]) / ih)
            pred_boxes.append((cls_id, xyxy.tolist(), norm_area))

    # Match GT to predictions (greedy, per-class, IoU >= 0.5)
    gt_matched = [False] * len(gt_boxes)
    pred_matched = [False] * len(pred_boxes)

    for gi, (g_cls, g_box) in enumerate(gt_boxes):
        best_iou = 0.0
        best_pi = -1
        # Convert GT from normalized to pixel coords for IoU comparison
        g_box_px = [g_box[0]*iw, g_box[1]*ih, g_box[2]*iw, g_box[3]*ih]
        for pi, (p_cls, p_box, _) in enumerate(pred_boxes):
            if pred_matched[pi] or p_cls != g_cls:
                continue
            iou = compute_iou(g_box_px, p_box)
            if iou > best_iou:
                best_iou = iou
                best_pi = pi
        if best_iou >= 0.5 and best_pi >= 0:
            gt_matched[gi] = True
            pred_matched[best_pi] = True
            tp_areas.append(gt_areas_raw[gi])
            tp_pred_areas.append(pred_boxes[best_pi][2])

    for gi, matched in enumerate(gt_matched):
        if not matched:
            fn_areas.append(gt_areas_raw[gi])

    for pi, matched in enumerate(pred_matched):
        if not matched:
            fp_areas.append(pred_boxes[pi][2])

print(f"\nMatching complete:")
print(f"  True Positives:  {len(tp_areas)}")
print(f"  False Negatives: {len(fn_areas)}")
print(f"  False Positives: {len(fp_areas)}")


In [ ]:
# ── Plot: Object size distributions ───────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: GT boxes — TP vs FN
ax = axes[0]
if tp_areas:
    ax.hist(np.array(tp_areas) * 100, bins=30, alpha=0.6, color="#27ae60",
            label=f"TP ({len(tp_areas)})", edgecolor="black")
if fn_areas:
    ax.hist(np.array(fn_areas) * 100, bins=30, alpha=0.6, color="#e74c3c",
            label=f"FN ({len(fn_areas)})", edgecolor="black")
ax.set_xlabel("Box Area (% of image)")
ax.set_ylabel("Count")
ax.set_title("Ground Truth: Detected (TP) vs Missed (FN)")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Predictions — TP vs FP
ax2 = axes[1]
if tp_pred_areas:
    ax2.hist(np.array(tp_pred_areas) * 100, bins=30, alpha=0.6, color="#27ae60",
             label=f"TP ({len(tp_pred_areas)})", edgecolor="black")
if fp_areas:
    ax2.hist(np.array(fp_areas) * 100, bins=30, alpha=0.6, color="#e74c3c",
             label=f"FP ({len(fp_areas)})", edgecolor="black")
ax2.set_xlabel("Box Area (% of image)")
ax2.set_ylabel("Count")
ax2.set_title("Predictions: Correct (TP) vs Hallucinated (FP)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Object Size vs Detection Quality", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary statistics
if fn_areas:
    print(f"Missed objects (FN): median area = {np.median(fn_areas)*100:.2f}% of image")
if tp_areas:
    print(f"Detected objects (TP): median area = {np.median(tp_areas)*100:.2f}% of image")
if fn_areas and tp_areas:
    ratio = np.median(fn_areas) / np.median(tp_areas)
    if ratio < 0.5:
        print(f"\nMissed objects are {1/ratio:.1f}x SMALLER than detected ones.")
        print("Options: (a) Higher imgsz, (b) Tiling, (c) More close-up training data, (d) Accept the limitation")
    else:
        print(f"\nNo strong size bias — missed objects are similar in size to detected ones.")

---
## Section 3: Per-Class Analysis

~5 minutes

Overall mAP hides critical information. Let's break it down:
- Which class does the model struggle with?
- Are hardhats being confused with persons (or vice versa)?
- What do the precision-recall curves look like?

In [ ]:
# ── Per-class recall with actual FN counts ───────────────────────────────────

class_tp = {}
class_fn = {}
class_fp = {}

for img_path in val_images:
    # Get image dimensions once
    pil_img = Image.open(img_path)
    iw, ih = pil_img.size

    label_path = val_labels_dir / f"{img_path.stem}.txt"
    gt_labels = parse_yolo_labels(label_path)
    gt_boxes = [(cls_id, yolo_to_xyxy(cx, cy, w, h)) for cls_id, cx, cy, w, h in gt_labels]

    results = model.predict(str(img_path), conf=0.25, device=DEVICE, verbose=False)
    pred_boxes = []
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            pred_boxes.append((cls_id, xyxy))

    # Match
    gt_matched = [False] * len(gt_boxes)
    pred_matched = [False] * len(pred_boxes)

    for gi, (g_cls, g_box) in enumerate(gt_boxes):
        best_iou, best_pi = 0.0, -1
        g_box_px = [g_box[0]*iw, g_box[1]*ih, g_box[2]*iw, g_box[3]*ih]
        for pi, (p_cls, p_box) in enumerate(pred_boxes):
            if pred_matched[pi] or p_cls != g_cls:
                continue
            iou = compute_iou(g_box_px, p_box)
            if iou > best_iou:
                best_iou = iou
                best_pi = pi
        if best_iou >= 0.5 and best_pi >= 0:
            gt_matched[gi] = True
            pred_matched[best_pi] = True
            class_tp[g_cls] = class_tp.get(g_cls, 0) + 1

    for gi, matched in enumerate(gt_matched):
        if not matched:
            g_cls = gt_boxes[gi][0]
            class_fn[g_cls] = class_fn.get(g_cls, 0) + 1

    for pi, matched in enumerate(pred_matched):
        if not matched:
            p_cls = pred_boxes[pi][0]
            class_fp[p_cls] = class_fp.get(p_cls, 0) + 1

# Print per-class results
print("=" * 60)
print("  PER-CLASS DETECTION RESULTS")
print("=" * 60)
print(f"  {'Class':<12} {'TP':>6} {'FN':>6} {'FP':>6} {'Recall':>8} {'Precision':>10}")
print(f"  {'-'*12} {'-'*6} {'-'*6} {'-'*6} {'-'*8} {'-'*10}")
for cls_id, name in sorted(class_names.items()):
    tp = class_tp.get(cls_id, 0)
    fn = class_fn.get(cls_id, 0)
    fp = class_fp.get(cls_id, 0)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    print(f"  {name:<12} {tp:>6} {fn:>6} {fp:>6} {recall:>8.3f} {precision:>10.3f}")
print("=" * 60)


In [ ]:
# ── Confusion matrix and PR curves (Ultralytics built-in) ─────────────────────

# Ultralytics saves plots during model.val()
# Let's re-run val with save_dir to capture them
import tempfile

val_dir = Path(tempfile.mkdtemp(prefix="nb03_val_"))

metrics_full = model.val(
    data=str(dataset_yaml),
    conf=0.25,
    iou=0.5,
    plots=True,
    save_dir=str(val_dir),
    verbose=False,
)

# Display the confusion matrix and PR curve
for plot_name in ["confusion_matrix_normalized.png", "confusion_matrix.png",
                  "PR_curve.png", "P_curve.png", "R_curve.png"]:
    plot_path = val_dir / plot_name
    if plot_path.exists():
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.imshow(Image.open(plot_path))
        ax.set_title(plot_name.replace(".png", "").replace("_", " ").title(), fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

### What to look for

- **Confusion matrix**: Are hardhats being confused with persons? Is the "background" row large (missed detections)?
- **PR curve**: Does precision drop sharply at high recall? Where is the sweet spot?
- **Per-class FN counts**: Absolute numbers matter more than percentages when the dataset is small

---
## Section 4: Confidence Threshold Exploration

~5 minutes

The default confidence threshold (0.25) is a tradeoff. Let's see how precision and recall change at different thresholds.

**For a safety-critical application like PPE compliance:**
- Would you prefer **higher precision** (fewer false alarms, but might miss violations)?
- Or **higher recall** (catch every violation, but more false alarms)?
- Why?

In [ ]:
# ── Evaluate at multiple confidence thresholds ────────────────────────────────

thresholds = [0.1, 0.25, 0.5, 0.7]
threshold_results = []

for conf_thresh in thresholds:
    m = model.val(
        data=str(dataset_yaml),
        conf=conf_thresh,
        iou=0.5,
        verbose=False,
    )
    threshold_results.append({
        "conf": conf_thresh,
        "mAP50": m.box.map50,
        "precision": np.mean(m.box.p) if len(m.box.p) > 0 else 0,
        "recall": np.mean(m.box.r) if len(m.box.r) > 0 else 0,
    })

# Display results
print("=" * 60)
print("  CONFIDENCE THRESHOLD EXPLORATION")
print("=" * 60)
print(f"  {'Conf':>6} {'mAP50':>8} {'Precision':>10} {'Recall':>8}")
print(f"  {'-'*6} {'-'*8} {'-'*10} {'-'*8}")
for r in threshold_results:
    print(f"  {r['conf']:>6.2f} {r['mAP50']:>8.3f} {r['precision']:>10.3f} {r['recall']:>8.3f}")
print("=" * 60)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
confs = [r["conf"] for r in threshold_results]
ax.plot(confs, [r["precision"] for r in threshold_results], "o-", color="#3498db",
        linewidth=2, markersize=8, label="Precision")
ax.plot(confs, [r["recall"] for r in threshold_results], "s-", color="#e74c3c",
        linewidth=2, markersize=8, label="Recall")
ax.plot(confs, [r["mAP50"] for r in threshold_results], "^-", color="#27ae60",
        linewidth=2, markersize=8, label="mAP50")
ax.set_xlabel("Confidence Threshold", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Precision / Recall / mAP50 vs Confidence Threshold", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### Think about this

For PPE compliance on a construction site:
- A **false negative** (missed non-compliant worker) = someone without a hardhat goes unnoticed = **safety risk**
- A **false positive** (flagging a compliant worker as non-compliant) = unnecessary alert = **annoyance**

In safety-critical applications, you typically want **high recall** even at the cost of some precision. A few false alarms are better than missing a real violation.

What confidence threshold would you choose for deployment? Write it down — you will use it in the compliance challenge.

---
## Section 5: The Compliance Challenge

~15 minutes | **THIS IS THE EXERCISE**

You have a model that detects **hardhats** and **persons**. The construction site manager wants to know:

> *"What percentage of workers are NOT wearing a hardhat?"*

Your task: **design an algorithm** that takes bounding box detections and outputs a compliance assessment.

### The Building Blocks

The cells below give you the raw ingredients. The algorithm design is up to you.

In [ ]:
# ── Building Block 1: Extract detections from a single image ─────────────────

# Pick a test image with known compliance (easy/ has workers all wearing hardhats)
test_image = sorted(IMAGES_DIR.glob("easy/*.jpg"))[0]  # easy_01.jpg
print(f"Test image: {test_image}")

# Run detection
results = model.predict(str(test_image), conf=0.25, device=DEVICE, verbose=False)

# Extract bounding boxes per class
person_boxes = []   # list of [x1, y1, x2, y2] in pixel coordinates
person_confs = []   # confidence score for each person
hardhat_boxes = []  # list of [x1, y1, x2, y2] in pixel coordinates
hardhat_confs = []  # confidence score for each hardhat

if results and results[0].boxes is not None:
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        xyxy = box.xyxy[0].cpu().numpy().tolist()
        conf_val = float(box.conf[0])
        if cls_id == 0:  # hardhat
            hardhat_boxes.append(xyxy)
            hardhat_confs.append(conf_val)
        elif cls_id == 1:  # person
            person_boxes.append(xyxy)
            person_confs.append(conf_val)

print(f"Detected: {len(person_boxes)} persons, {len(hardhat_boxes)} hardhats")
print(f"\nPerson boxes (pixel coords):")
for i, (box, conf) in enumerate(zip(person_boxes, person_confs)):
    print(f"  Person {i}: [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]  conf={conf:.3f}")
print(f"\nHardhat boxes (pixel coords):")
for i, (box, conf) in enumerate(zip(hardhat_boxes, hardhat_confs)):
    print(f"  Hardhat {i}: [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]  conf={conf:.3f}")


In [ ]:
# ── Building Block 2: Visualize detections on an image ───────────────────────

def show_detections(image_path, person_boxes, hardhat_boxes,
                    person_labels=None, figsize=(12, 8)):
    """Draw person and hardhat boxes on an image.

    Args:
        image_path: Path to the image file.
        person_boxes: List of [x1, y1, x2, y2] person bounding boxes.
        hardhat_boxes: List of [x1, y1, x2, y2] hardhat bounding boxes.
        person_labels: Optional list of strings (one per person), e.g. "COMPLIANT".
    """
    img = np.array(Image.open(image_path))
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)

    # Draw hardhat boxes (green)
    for hx1, hy1, hx2, hy2 in hardhat_boxes:
        rect = plt.Rectangle((hx1, hy1), hx2 - hx1, hy2 - hy1,
                             linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(hx1, hy1 - 4, "hardhat", color="lime", fontsize=7,
                bbox=dict(facecolor="black", alpha=0.7, pad=1))

    # Draw person boxes
    for i, (px1, py1, px2, py2) in enumerate(person_boxes):
        label = person_labels[i] if person_labels else "person"
        if "NON" in label.upper() or "UNSAFE" in label.upper():
            color = "red"
        elif "COMPLIANT" in label.upper() or "SAFE" in label.upper():
            color = "green"
        else:
            color = "cyan"
        rect = plt.Rectangle((px1, py1), px2 - px1, py2 - py1,
                             linewidth=3, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(px1, py2 + 12, label, color=color, fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.7, pad=2))

    ax.axis("off")
    plt.tight_layout()
    plt.show()


# Visualize the raw detections
show_detections(test_image, person_boxes, hardhat_boxes)

In [ ]:
# ── Building Block 3: IoU helper (already defined above) ─────────────────────
# compute_iou(box_a, box_b) is available from Section 2
# Both boxes should be [x1, y1, x2, y2] in the same coordinate system

# Quick test:
box_a = [10, 10, 50, 50]
box_b = [30, 30, 70, 70]
print(f"IoU of overlapping boxes: {compute_iou(box_a, box_b):.3f}")

box_c = [100, 100, 150, 150]
print(f"IoU of non-overlapping boxes: {compute_iou(box_a, box_c):.3f}")

### The Challenge

Now it is your turn. Given `person_boxes` and `hardhat_boxes`, design an algorithm that:

1. For each person, determines: **COMPLIANT** (wearing a hardhat) or **NON-COMPLIANT** (not wearing one)
2. Computes the overall **compliance rate** (% of workers wearing hardhats)

**Hints** (try to think through it yourself before reading):
- Think about spatial relationships. A hardhat that is detected far below a person's feet probably does not belong to that person.
- Where on a person's body would you expect to find a hardhat? What sub-region of the person bounding box?
- How close does a hardhat need to be to a person to count as "wearing"? What metric would you use?
- What if two persons are close together and one hardhat is between them?

There is no single right answer. A simple approach can work well.

> **Stuck?** Ask your AI copilot: *"How can I determine if a detected person is wearing a hardhat using only bounding box coordinates? I have person boxes and hardhat boxes in [x1,y1,x2,y2] format."*

In [ ]:
# ======================================================================
# YOUR COMPLIANCE ALGORITHM HERE
# ======================================================================
#
# Input:
#   person_boxes  - list of [x1, y1, x2, y2] bounding boxes (pixel coords)
#   hardhat_boxes - list of [x1, y1, x2, y2] bounding boxes (pixel coords)
#
# Output:
#   compliance_labels - list of strings, one per person: "COMPLIANT" or "NON-COMPLIANT"
#   compliance_rate   - float, fraction of workers wearing hardhats
#
# You can use compute_iou(box_a, box_b) which is already defined.
#
# ======================================================================

def check_compliance(person_boxes, hardhat_boxes):
    """Determine compliance status for each detected person.

    Args:
        person_boxes: List of [x1, y1, x2, y2] person bounding boxes.
        hardhat_boxes: List of [x1, y1, x2, y2] hardhat bounding boxes.

    Returns:
        Tuple of (compliance_labels, compliance_rate) where:
          - compliance_labels is a list of "COMPLIANT" / "NON-COMPLIANT" strings
          - compliance_rate is the fraction of compliant persons (0.0 to 1.0)
    """
    compliance_labels = []

    for person in person_boxes:
        # YOUR LOGIC HERE
        # Decide if this person is wearing a hardhat
        is_compliant = False  # TODO: replace with your algorithm

        compliance_labels.append("COMPLIANT" if is_compliant else "NON-COMPLIANT")

    compliant_count = sum(1 for l in compliance_labels if l == "COMPLIANT")
    compliance_rate = compliant_count / len(person_boxes) if person_boxes else 0.0

    return compliance_labels, compliance_rate

In [ ]:
# ── Test your algorithm on the easy image ────────────────────────────────────

labels, rate = check_compliance(person_boxes, hardhat_boxes)

print(f"Image: {test_image.name}")
print(f"Compliance rate: {rate:.0%}")
for i, (label, box) in enumerate(zip(labels, person_boxes)):
    print(f"  Person {i}: {label}")

# Visualize with compliance labels
show_detections(test_image, person_boxes, hardhat_boxes, person_labels=labels)

In [ ]:
# ── Test on multiple images across categories ────────────────────────────────

test_categories = ["easy", "mixed_compliance", "cctv", "close_up", "warehouse"]
conf_threshold = 0.25  # Use the threshold you chose in Section 4

all_compliant = 0
all_persons = 0

for cat_name in test_categories:
    cat_dir = IMAGES_DIR / cat_name
    if not cat_dir.exists():
        continue
    images = sorted(cat_dir.glob("*.jpg"))[:2]  # 2 images per category
    for img_path in images:
        results = model.predict(str(img_path), conf=conf_threshold, device=DEVICE, verbose=False)

        p_boxes, h_boxes = [], []
        if results and results[0].boxes is not None:
            for box in results[0].boxes:
                cls_id = int(box.cls[0])
                xyxy = box.xyxy[0].cpu().numpy().tolist()
                if cls_id == 0:
                    h_boxes.append(xyxy)
                elif cls_id == 1:
                    p_boxes.append(xyxy)

        if not p_boxes:
            print(f"  {cat_name}/{img_path.name}: no persons detected")
            continue

        labels, rate = check_compliance(p_boxes, h_boxes)
        n_compliant = sum(1 for l in labels if l == "COMPLIANT")
        all_compliant += n_compliant
        all_persons += len(p_boxes)

        print(f"  {cat_name}/{img_path.name}: "
              f"{n_compliant}/{len(p_boxes)} compliant ({rate:.0%}) | "
              f"{len(h_boxes)} hardhats detected")

        # Show the first image from each category with compliance labels
        if img_path == images[0]:
            show_detections(img_path, p_boxes, h_boxes, person_labels=labels)

if all_persons > 0:
    print(f"\nOverall: {all_compliant}/{all_persons} workers compliant ({all_compliant/all_persons:.0%})")


### Evaluate Your Algorithm

Look at the visualizations above:
- Are the compliance labels correct for the images you can see?
- Does the algorithm handle edge cases? (Workers far away, multiple workers close together)
- What happens when the model misses a hardhat (false negative)? What happens when it hallucinates one (false positive)?

Model errors propagate into compliance errors. A model with 80% hardhat recall will systematically **over-estimate non-compliance** because it misses real hardhats.

---
## Section 6: Speed Benchmark

~3 minutes

Production requirement: **30+ FPS** (< 33ms per image).

Let's compare your trained YOLO26n against the YOLOe-26n foundation model you started with.

In [ ]:
# ── Inference speed benchmark ────────────────────────────────────────────────

# DEVICE was defined in the setup cell above
print(f"Benchmarking on: {DEVICE}")

# Get a batch of validation images for timing
bench_images = val_images[:10]  # 10 images

# ---- Benchmark: Trained YOLO26n (student) ----
student_model = YOLO(str(MODEL_PATH))

# Warmup
for _ in range(3):
    student_model.predict(str(bench_images[0]), device=DEVICE, verbose=False)

t0 = time.perf_counter()
for img_path in bench_images:
    student_model.predict(str(img_path), device=DEVICE, verbose=False)
student_time = (time.perf_counter() - t0) / len(bench_images) * 1000  # ms/image

# ---- Benchmark: YOLOe-26n (foundation model) ----
yoloe_path = WORKSHOP_DIR / "notebooks" / "yoloe-26n-seg.pt"
yoloe_time = None

if yoloe_path.exists():
    yoloe_model = YOLO(str(yoloe_path))
    yoloe_model.set_classes(["hard hat", "person"])

    # Warmup
    for _ in range(3):
        yoloe_model.predict(str(bench_images[0]), device=DEVICE, verbose=False)

    t0 = time.perf_counter()
    for img_path in bench_images:
        yoloe_model.predict(str(img_path), device=DEVICE, verbose=False)
    yoloe_time = (time.perf_counter() - t0) / len(bench_images) * 1000
else:
    print(f"YOLOe model not found at {yoloe_path} -- skipping comparison")

# ---- Results ----
print("\n" + "=" * 60)
print("  INFERENCE SPEED BENCHMARK")
print("=" * 60)
print(f"  {'Model':<30} {'ms/image':>10} {'FPS':>8}")
print(f"  {'-'*30} {'-'*10} {'-'*8}")
print(f"  {'YOLO26n (trained student)':<30} {student_time:>10.1f} {1000/student_time:>8.0f}")
if yoloe_time:
    print(f"  {'YOLOe-26n (foundation)':<30} {yoloe_time:>10.1f} {1000/yoloe_time:>8.0f}")
    speedup = yoloe_time / student_time
    print(f"\n  Speedup: {speedup:.1f}x faster with trained student")

if student_time < 33:
    print(f"\n  Your model runs at {1000/student_time:.0f} FPS -- meets the 30+ FPS production target.")
else:
    print(f"\n  Your model runs at {1000/student_time:.0f} FPS -- below the 30 FPS target.")
    print(f"  Consider: smaller imgsz, GPU acceleration, or model pruning.")
print("=" * 60)


---
## Section 7: Model Comparison Table

Bring all your results together in one table.

In [ ]:
# ── Utility: Build comparison table ──────────────────────────────────────────

def build_comparison_table(results_list):
    """Build a comparison table from a list of result dictionaries.

    Each dict should have: name, map50, speed_ms.
    Optionally: map50_95, compliance_rate.
    """
    rows = []
    for r in results_list:
        rows.append({
            "Model": r.get("name", "Unknown"),
            "mAP50": f"{r['map50']:.3f}" if r.get("map50") is not None else "-",
            "mAP50-95": f"{r['map50_95']:.3f}" if r.get("map50_95") is not None else "-",
            "Speed (ms)": f"{r['speed_ms']:.1f}" if r.get("speed_ms") is not None else "-",
            "FPS": f"{1000/r['speed_ms']:.0f}" if r.get("speed_ms") is not None else "-",
            "Compliance Rate": f"{r['compliance_rate']:.0%}" if r.get("compliance_rate") is not None else "-",
        })
    return pd.DataFrame(rows)

In [ ]:
# ======================================================================
# FILL IN YOUR RESULTS
# ======================================================================
# Replace the placeholder values with your actual measurements
# from the evaluations above.

my_results = [
    {
        "name": "YOLOe-26n (zero-shot)",
        "map50": None,        # From NB01 Phase 1 evaluation (or leave None)
        "speed_ms": yoloe_time if yoloe_time else None,
    },
    {
        "name": "YOLO26n (my training)",
        "map50": metrics.box.map50,
        "map50_95": metrics.box.map,
        "speed_ms": student_time,
        "compliance_rate": all_compliant / all_persons if all_persons > 0 else None,
    },
]

df = build_comparison_table(my_results)
print(df.to_string(index=False))

---
## Section 8: Discussion & Reflection

~5 minutes

### Questions to Discuss

1. **Pipeline impact**: What was the biggest factor in your model's performance -- the auto-labeler, the labeling prompt, the data curation, or the training configuration? Why?

2. **More data vs better labels**: You saw the saturating curve in the theory block -- adding more images with noisy labels barely helps. What is the right investment: more images or better labeling? When would you choose one over the other?

3. **Business metrics vs model metrics**: Your model has a certain mAP50. Your compliance algorithm has a certain accuracy. When might a model with **lower** mAP50 actually produce **better** compliance accuracy? (Hint: think about which class matters more for compliance.)

4. **Deployment edge cases**: If you deployed this on a real construction site, what would worry you? Think about:
   - Camera angles (CCTV vs eye-level)
   - Lighting (dawn, dusk, harsh shadows)
   - Occlusion (workers behind equipment)
   - Scale (workers far from camera vs close-up)

5. **The compliance algorithm**: How sensitive is your algorithm to its parameters? What happens if you change the threshold(s) you chose? Is there a single "best" parameter set, or does it depend on the deployment context?

### Key Takeaways (from our reference experiments)

| Lesson | Evidence |
|--------|----------|
| Label quality > data quantity | Saturating curve: mAP50 asymptote ~0.53 with noisy labels, broken to ~0.63 with curated labels |
| Prompt engineering is first-order | Different wording finds different objects. SAM3 HF takes one concept per call — always test prompt variations (e.g., "helmet" vs "hard hat" vs "safety helmet") before committing to a labeling run. |
| Business metrics != model metrics | mAP50 can be high while compliance accuracy is low (or vice versa) |
| Foundation models -> distillation | SAM3/YOLOE label data -> train YOLO26n for 30+ FPS production deployment |
| 2-class + post-processing > 3-class | Detecting absences is fundamentally harder than detecting objects |
| Inspect before training | In our reference run, ~35% of auto-labels were noise (tiny labels) |

### Bonus Explorations

If you have time:
- **SAM3 vs Qwen 3.5**: Run `poc_qwen35_vl_grounding.ipynb` to compare two foundation model auto-labelers
- **Resolution tradeoff**: Try different image sizes (imgsz=320, 640, 1024) and measure the speed/accuracy tradeoff
- **Prompt engineering**: Re-run auto-labeling with a different prompt and compare label quality
- **FiftyOne deep dive**: Run `demo_04_fiftyone_data_analysis.ipynb` with `DOMAIN="ppe"` for data-centric analysis


---
## Section 9: Phase 5 Observations

Write your final observations here:

**Detection performance:**
- mAP50: ___
- Strongest class: ___
- Weakest class: ___
- Biggest source of errors (size? class? scene type?): ___

**Compliance algorithm:**
- Approach you used: ___
- Overall compliance rate measured: ___
- Failure modes you observed: ___
- What would you change to improve it: ___

**Speed:**
- Student model FPS: ___
- Meets 30+ FPS target? ___
- Speedup over foundation model: ___x

**Biggest insight from the full workshop pipeline:**
- ___

**What would you do differently with more time:**
- [ ] ...
- [ ] ...
- [ ] ...